In [2]:
import os
import numpy as np
import pandas as pd

print("Baseline modeling environment ready")

Baseline modeling environment ready


In [3]:
ml_file = "../data/processed/features/ml_dataset.csv"

ml_dataset = pd.read_csv(ml_file)

print("Shape:", ml_dataset.shape)
print("Patients:", ml_dataset["patient_id"].nunique())

Shape: (11109, 17)
Patients: 44


In [4]:
print(ml_dataset.columns.tolist())

['patient_id', 'mean_rr', 'sdnn', 'rmssd', 'pnn50', 'lf_power', 'hf_power', 'lf_hf', 'window_id', 'rr_count', 'sdnn_trend', 'rmssd_trend', 'pnn50_trend', 'age', 'sex', 'nyha_class', 'risk_label']


In [5]:
feature_columns = [
    "mean_rr",
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_power",
    "hf_power",
    "lf_hf",
    "rr_count",
    "sdnn_trend",
    "rmssd_trend",
    "pnn50_trend"
]

X = ml_dataset[feature_columns]
y = ml_dataset["risk_label"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11109, 11)
y shape: (11109,)


In [6]:
from sklearn.model_selection import train_test_split

patient_labels = (
    ml_dataset[
        ["patient_id", "risk_label"]
    ]
    .drop_duplicates()
)

print(patient_labels.shape)
print(patient_labels["risk_label"].value_counts())

(44, 2)
risk_label
1    32
0    12
Name: count, dtype: int64


In [7]:
train_patients, test_patients = train_test_split(
    patient_labels,
    test_size=0.20,
    random_state=42,
    stratify=patient_labels["risk_label"]
)

print("Training patients:", len(train_patients))
print("Testing patients:", len(test_patients))

Training patients: 35
Testing patients: 9


In [8]:
print("Training class distribution:")
print(train_patients["risk_label"].value_counts())

print("\nTesting class distribution:")
print(test_patients["risk_label"].value_counts())

Training class distribution:
risk_label
1    25
0    10
Name: count, dtype: int64

Testing class distribution:
risk_label
1    7
0    2
Name: count, dtype: int64


In [9]:
overlap = set(train_patients["patient_id"]) & set(
    test_patients["patient_id"]
)

print("Patient overlap:", overlap)

Patient overlap: set()


In [10]:
train_ids = set(train_patients["patient_id"])
test_ids = set(test_patients["patient_id"])

train_df = ml_dataset[
    ml_dataset["patient_id"].isin(train_ids)
].copy()

test_df = ml_dataset[
    ml_dataset["patient_id"].isin(test_ids)
].copy()

print("Training windows:", len(train_df))
print("Testing windows:", len(test_df))

Training windows: 8940
Testing windows: 2169


In [11]:
print(
    "Training patients:",
    train_df["patient_id"].nunique()
)

print(
    "Testing patients:",
    test_df["patient_id"].nunique()
)

Training patients: 35
Testing patients: 9


In [12]:
X_train = train_df[feature_columns]
y_train = train_df["risk_label"]

X_test = test_df[feature_columns]
y_test = test_df["risk_label"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (8940, 11)
X_test: (2169, 11)


In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print("Scaling complete")

Scaling complete


In [14]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

print("Logistic Regression trained successfully")

Logistic Regression trained successfully


In [15]:
y_pred = logistic_model.predict(X_test_scaled)

y_prob = logistic_model.predict_proba(
    X_test_scaled
)[:, 1]

print("Predictions generated")

Predictions generated


In [16]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Results")
print("---------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("AUROC    :", auc)

Logistic Regression Results
---------------------------
Accuracy : 0.5850622406639004
Precision: 0.7424522489217499
Recall   : 0.7142857142857143
F1 Score : 0.7280966767371602
AUROC    : 0.4144187796845292


In [17]:
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)


Confusion Matrix:
[[  64  418]
 [ 482 1205]]


In [18]:
print("Training patients:", len(train_patients))
print("Testing patients:", len(test_patients))

print("Training class distribution:")
print(train_patients["risk_label"].value_counts())

print("Testing class distribution:")
print(test_patients["risk_label"].value_counts())

print("Patient overlap:", overlap)

Training patients: 35
Testing patients: 9
Training class distribution:
risk_label
1    25
0    10
Name: count, dtype: int64
Testing class distribution:
risk_label
1    7
0    2
Name: count, dtype: int64
Patient overlap: set()


In [19]:
print(
    test_df[
        ["patient_id", "risk_label"]
    ].drop_duplicates().sort_values("patient_id")
)

      patient_id  risk_label
240        chf02           1
470        chf03           1
1665       chf08           1
4860      chf206           1
5111      chf207           1
5626      chf209           1
6890      chf214           0
7630      chf217           0
10563     chf228           1


In [20]:
print(
    test_df.groupby(
        ["patient_id", "risk_label"]
    ).size()
)

patient_id  risk_label
chf02       1             230
chf03       1             240
chf08       1             240
chf206      1             251
chf207      1             227
chf209      1             226
chf214      0             203
chf217      0             279
chf228      1             273
dtype: int64


In [21]:
test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

test_results["predicted_probability"] = y_prob
test_results["predicted_class"] = y_pred

In [22]:
patient_results = (
    test_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
       #predicted_class=("predicted_class", "mean"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [23]:
patient_results["predicted_class"] = (
    patient_results["mean_probability"] >= 0.5
).astype(int)

In [24]:
print(patient_results)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          1.000000            1.000000           230   
1      chf03           1          0.699295            0.713641           240   
2      chf08           1          0.491855            0.508818           240   
3     chf206           1          0.616369            0.637882           251   
4     chf207           1          0.997221            0.997793           227   
5     chf209           1          0.381264            0.384402           226   
6     chf214           0          0.986767            0.987077           203   
7     chf217           0          0.639977            0.633862           279   
8     chf228           1          0.495592            0.507719           273   

   predicted_class  
0                1  
1                1  
2                0  
3                1  
4                1  
5                0  
6                1  
7                1  
8         

In [25]:
patient_y_true = patient_results["risk_label"]

patient_y_pred = patient_results["predicted_class"]

patient_y_prob = patient_results["mean_probability"]

In [26]:
print("\nPatient-level confusion matrix:")
print(
    confusion_matrix(
        patient_y_true,
        patient_y_pred
    )
)


Patient-level confusion matrix:
[[0 2]
 [3 4]]


In [27]:
test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

test_results["predicted_probability"] = y_prob
#test_results["predicted_class"] = y_pred

In [28]:
patient_y_true = patient_results["risk_label"]
patient_y_pred = patient_results["predicted_class"]
patient_y_prob = patient_results["mean_probability"]

In [29]:
print("Patient-level Accuracy:",
      accuracy_score(patient_y_true, patient_y_pred))

print("Patient-level Precision:",
      precision_score(patient_y_true, patient_y_pred, zero_division=0))

print("Patient-level Recall:",
      recall_score(patient_y_true, patient_y_pred, zero_division=0))

print("Patient-level F1:",
      f1_score(patient_y_true, patient_y_pred, zero_division=0))

print("Patient-level AUROC:",
      roc_auc_score(patient_y_true, patient_y_prob))

print("\nPatient-level confusion matrix:")
print(
    confusion_matrix(
        patient_y_true,
        patient_y_pred
    )
)

Patient-level Accuracy: 0.4444444444444444
Patient-level Precision: 0.6666666666666666
Patient-level Recall: 0.5714285714285714
Patient-level F1: 0.6153846153846154
Patient-level AUROC: 0.3571428571428571

Patient-level confusion matrix:
[[0 2]
 [3 4]]


In [30]:
print(patient_results)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          1.000000            1.000000           230   
1      chf03           1          0.699295            0.713641           240   
2      chf08           1          0.491855            0.508818           240   
3     chf206           1          0.616369            0.637882           251   
4     chf207           1          0.997221            0.997793           227   
5     chf209           1          0.381264            0.384402           226   
6     chf214           0          0.986767            0.987077           203   
7     chf217           0          0.639977            0.633862           279   
8     chf228           1          0.495592            0.507719           273   

   predicted_class  
0                1  
1                1  
2                0  
3                1  
4                1  
5                0  
6                1  
7                1  
8         

In [31]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(
    X_train,
    y_train
)

print("Random Forest trained successfully")

Random Forest trained successfully


In [32]:
rf_y_pred = rf_model.predict(X_test)

rf_y_prob = rf_model.predict_proba(
    X_test
)[:, 1]

print("Predictions generated")

Predictions generated


In [33]:
print("Random Forest — Window-level Results")
print("------------------------------------")

print("Accuracy :",
      accuracy_score(y_test, rf_y_pred))

print("Precision:",
      precision_score(y_test, rf_y_pred, zero_division=0))

print("Recall   :",
      recall_score(y_test, rf_y_pred, zero_division=0))

print("F1 Score :",
      f1_score(y_test, rf_y_pred, zero_division=0))

print("AUROC    :",
      roc_auc_score(y_test, rf_y_prob))

Random Forest — Window-level Results
------------------------------------
Accuracy : 0.764407561088059
Precision: 0.7747663551401869
Recall   : 0.982809721398933
F1 Score : 0.866475045727724
AUROC    : 0.08799226203799128


In [34]:
rf_test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

rf_test_results["predicted_probability"] = rf_y_prob

In [35]:
rf_patient_results = (
    rf_test_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [36]:
rf_patient_results["predicted_class"] = (
    rf_patient_results["mean_probability"] >= 0.5
).astype(int)

In [37]:
print(rf_patient_results)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          0.986261            0.983333           230   
1      chf03           1          0.882000            0.876667           240   
2      chf08           1          0.670750            0.670000           240   
3     chf206           1          0.583519            0.606667           251   
4     chf207           1          0.984684            0.983333           227   
5     chf209           1          0.635413            0.636667           226   
6     chf214           0          0.994811            0.996667           203   
7     chf217           0          0.985448            0.986667           279   
8     chf228           1          0.861917            0.863333           273   

   predicted_class  
0                1  
1                1  
2                1  
3                1  
4                1  
5                1  
6                1  
7                1  
8         

In [38]:
rf_patient_y_true = rf_patient_results["risk_label"]
rf_patient_y_pred = rf_patient_results["predicted_class"]
rf_patient_y_prob = rf_patient_results["mean_probability"]

In [39]:
print("Random Forest — Patient-level Results")
print("-------------------------------------")

print("Accuracy :",
      accuracy_score(
          rf_patient_y_true,
          rf_patient_y_pred
      ))

print("Precision:",
      precision_score(
          rf_patient_y_true,
          rf_patient_y_pred,
          zero_division=0
      ))

print("Recall   :",
      recall_score(
          rf_patient_y_true,
          rf_patient_y_pred,
          zero_division=0
      ))

print("F1 Score :",
      f1_score(
          rf_patient_y_true,
          rf_patient_y_pred,
          zero_division=0
      ))

print("AUROC    :",
      roc_auc_score(
          rf_patient_y_true,
          rf_patient_y_prob
      ))

Random Forest — Patient-level Results
-------------------------------------
Accuracy : 0.7777777777777778
Precision: 0.7777777777777778
Recall   : 1.0
F1 Score : 0.875
AUROC    : 0.07142857142857142


In [40]:
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        rf_patient_y_true,
        rf_patient_y_pred
    )
)


Confusion Matrix:
[[0 2]
 [0 7]]


In [41]:
rf_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": rf_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(rf_importance)

        feature  importance
10  pnn50_trend    0.325637
9   rmssd_trend    0.271522
8    sdnn_trend    0.236236
6         lf_hf    0.042864
0       mean_rr    0.030792
7      rr_count    0.022315
4      lf_power    0.021733
2         rmssd    0.015503
3         pnn50    0.013646
1          sdnn    0.010405
5      hf_power    0.009347


In [42]:
#Random forest without trends

In [43]:
baseline_features = [
    "mean_rr",
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_power",
    "hf_power",
    "lf_hf",
    "rr_count"
]

In [44]:
X_train_base = train_df[baseline_features]
X_test_base = test_df[baseline_features]

In [45]:
rf_base = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_base.fit(X_train_base, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [46]:
rf_base_prob = rf_base.predict_proba(X_test_base)[:, 1]
rf_base_pred = rf_base.predict(X_test_base)

In [47]:
rf_base_results = test_df[
    ["patient_id", "risk_label"]
].copy()

rf_base_results["predicted_probability"] = rf_base_prob

In [48]:
rf_base_patient = (
    rf_base_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [49]:
rf_base_patient["predicted_class"] = (
    rf_base_patient["mean_probability"] >= 0.5
).astype(int)

In [50]:
rf_base_true = rf_base_patient["risk_label"]
rf_base_pred = rf_base_patient["predicted_class"]
rf_base_prob = rf_base_patient["mean_probability"]

In [51]:
print("Random Forest — HRV Only")
print("------------------------")

print("Accuracy :",
      accuracy_score(rf_base_true, rf_base_pred))

print("Precision:",
      precision_score(rf_base_true, rf_base_pred, zero_division=0))

print("Recall   :",
      recall_score(rf_base_true, rf_base_pred, zero_division=0))

print("F1 Score :",
      f1_score(rf_base_true, rf_base_pred, zero_division=0))

print("AUROC    :",
      roc_auc_score(rf_base_true, rf_base_prob))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        rf_base_true,
        rf_base_pred
    )
)

Random Forest — HRV Only
------------------------
Accuracy : 0.6666666666666666
Precision: 0.75
Recall   : 0.8571428571428571
F1 Score : 0.8
AUROC    : 0.42857142857142855

Confusion Matrix:
[[0 2]
 [1 6]]


In [52]:
print(rf_base_patient)

  patient_id  risk_label  mean_probability  median_probability  window_count  \
0      chf02           1          0.583261            0.693333           230   
1      chf03           1          0.805194            0.910000           240   
2      chf08           1          0.625028            0.636667           240   
3     chf206           1          0.582337            0.576667           251   
4     chf207           1          0.384258            0.300000           227   
5     chf209           1          0.839277            0.880000           226   
6     chf214           0          0.689442            0.746667           203   
7     chf217           0          0.620741            0.640000           279   
8     chf228           1          0.640781            0.666667           273   

   predicted_class  
0                1  
1                1  
2                1  
3                1  
4                0  
5                1  
6                1  
7                1  
8         

In [53]:
import xgboost

print("XGBoost version:", xgboost.__version__)

XGBoost version: 3.2.0


In [54]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    y_train
)

print("XGBoost trained successfully")

XGBoost trained successfully


In [55]:
xgb_y_pred = xgb_model.predict(X_test)

xgb_y_prob = xgb_model.predict_proba(
    X_test
)[:, 1]

print("Predictions generated")

Predictions generated


In [56]:
print("XGBoost — Window-level Results")
print("------------------------------")

print("Accuracy :",
      accuracy_score(y_test, xgb_y_pred))

print("Precision:",
      precision_score(y_test, xgb_y_pred, zero_division=0))

print("Recall   :",
      recall_score(y_test, xgb_y_pred, zero_division=0))

print("F1 Score :",
      f1_score(y_test, xgb_y_pred, zero_division=0))

print("AUROC    :",
      roc_auc_score(y_test, xgb_y_prob))

XGBoost — Window-level Results
------------------------------
Accuracy : 0.7777777777777778
Precision: 0.7777777777777778
Recall   : 1.0
F1 Score : 0.875
AUROC    : 0.16999842584371086


In [57]:
xgb_test_results = test_df[
    ["patient_id", "risk_label"]
].copy()

xgb_test_results["predicted_probability"] = xgb_y_prob

In [58]:
xgb_patient_results = (
    xgb_test_results
    .groupby(["patient_id", "risk_label"])
    .agg(
        mean_probability=("predicted_probability", "mean"),
        median_probability=("predicted_probability", "median"),
        window_count=("predicted_probability", "count")
    )
    .reset_index()
)

In [59]:
xgb_patient_results["predicted_class"] = (
    xgb_patient_results["mean_probability"] >= 0.5
).astype(int)

In [60]:
xgb_patient_y_true = xgb_patient_results["risk_label"]
xgb_patient_y_pred = xgb_patient_results["predicted_class"]
xgb_patient_y_prob = xgb_patient_results["mean_probability"]

In [61]:
print("XGBoost — Patient-level Results")
print("--------------------------------")

print("Accuracy :",
      accuracy_score(
          xgb_patient_y_true,
          xgb_patient_y_pred
      ))

print("Precision:",
      precision_score(
          xgb_patient_y_true,
          xgb_patient_y_pred,
          zero_division=0
      ))

print("Recall   :",
      recall_score(
          xgb_patient_y_true,
          xgb_patient_y_pred,
          zero_division=0
      ))

print("F1 Score :",
      f1_score(
          xgb_patient_y_true,
          xgb_patient_y_pred,
          zero_division=0
      ))

print("AUROC    :",
      roc_auc_score(
          xgb_patient_y_true,
          xgb_patient_y_prob
      ))

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        xgb_patient_y_true,
        xgb_patient_y_pred
    )
)

XGBoost — Patient-level Results
--------------------------------
Accuracy : 0.7777777777777778
Precision: 0.7777777777777778
Recall   : 1.0
F1 Score : 0.875
AUROC    : 0.2857142857142857

Confusion Matrix:
[[0 2]
 [0 7]]


In [62]:
xgb_importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": xgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print(xgb_importance)

        feature  importance
10  pnn50_trend    0.320437
8    sdnn_trend    0.211731
6         lf_hf    0.191194
9   rmssd_trend    0.160247
0       mean_rr    0.041035
1          sdnn    0.028856
3         pnn50    0.016424
4      lf_power    0.010863
7      rr_count    0.010331
5      hf_power    0.005551
2         rmssd    0.003332


In [63]:
baseline_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest - HRV Only",
        "Random Forest - HRV + Trends",
        "XGBoost - HRV + Trends"
    ],
    "Accuracy": [
        0.444444,
        0.666667,
        0.777778,
        0.777778
    ],
    "Precision": [
        0.666667,
        0.750000,
        0.777778,
        0.777778
    ],
    "Recall": [
        0.571429,
        0.857143,
        1.000000,
        1.000000
    ],
    "F1": [
        0.615385,
        0.800000,
        0.875000,
        0.875000
    ],
    "AUROC": [
        0.357143,
        0.428571,
        0.071429,
        0.285714
    ]
})

baseline_results

,Model,Accuracy,Precision,Recall,F1,AUROC
0,Logistic Regression,0.444444,0.666667,0.571429,0.615385,0.357143
1,Random Forest - HRV Only,0.666667,0.750000,0.857143,0.800000,0.428571
2,Random Forest - HRV + Trends,0.777778,0.777778,1.000000,0.875000,0.071429
3,XGBoost - HRV + Trends,0.777778,0.777778,1.000000,0.875000,0.285714


In [64]:
from sklearn.model_selection import GroupKFold

In [65]:
print("X shape:", X.shape)
print("y shape:", y.shape)

patient_ids = ml_dataset["patient_id"]

print("Patient groups:", len(patient_ids))
print("Unique patients:", patient_ids.nunique())

X shape: (11109, 11)
y shape: (11109,)
Patient groups: 11109
Unique patients: 44


In [66]:
groups = ml_dataset["patient_id"]

print("Total samples:", len(X))
print("Total patients:", groups.nunique())

Total samples: 11109
Total patients: 44


In [67]:
patient_label_check = (
    ml_dataset[
        ["patient_id", "risk_label"]
    ]
    .drop_duplicates()
)

print(patient_label_check.shape)

print(
    patient_label_check["risk_label"].value_counts()
)

print(
    patient_label_check.groupby("patient_id")["risk_label"].nunique().max()
)

(44, 2)
risk_label
1    32
0    12
Name: count, dtype: int64
1


In [68]:
group_kfold = GroupKFold(n_splits=5)

folds = list(
    group_kfold.split(
        X,
        y,
        groups=groups
    )
)

print("Number of folds:", len(folds))

Number of folds: 5


In [69]:
for fold, (train_idx, val_idx) in enumerate(folds, start=1):

    train_patients = set(
        groups.iloc[train_idx]
    )

    val_patients = set(
        groups.iloc[val_idx]
    )

    overlap = train_patients.intersection(
        val_patients
    )

    print(
        f"Fold {fold}: "
        f"Train patients = {len(train_patients)}, "
        f"Validation patients = {len(val_patients)}, "
        f"Overlap = {len(overlap)}"
    )

Fold 1: Train patients = 35, Validation patients = 9, Overlap = 0
Fold 2: Train patients = 35, Validation patients = 9, Overlap = 0
Fold 3: Train patients = 36, Validation patients = 8, Overlap = 0
Fold 4: Train patients = 35, Validation patients = 9, Overlap = 0
Fold 5: Train patients = 35, Validation patients = 9, Overlap = 0


In [70]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [71]:
def evaluate_cv_model(model, X, y, groups, folds):

    fold_results = []

    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):

        X_train_cv = X.iloc[train_idx]
        X_val_cv = X.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model.fit(X_train_cv, y_train_cv)

        y_pred_cv = model.predict(X_val_cv)
        y_prob_cv = model.predict_proba(X_val_cv)[:, 1]

        result = {
            "fold": fold_number,
            "accuracy": accuracy_score(y_val_cv, y_pred_cv),
            "precision": precision_score(
                y_val_cv,
                y_pred_cv,
                zero_division=0
            ),
            "recall": recall_score(
                y_val_cv,
                y_pred_cv,
                zero_division=0
            ),
            "f1": f1_score(
                y_val_cv,
                y_pred_cv,
                zero_division=0
            ),
            "auroc": roc_auc_score(
                y_val_cv,
                y_prob_cv
            )
        }

        fold_results.append(result)

    return pd.DataFrame(fold_results)

In [72]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [73]:
lr_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

In [74]:
lr_cv_results = evaluate_cv_model(
    lr_cv,
    X,
    y,
    groups,
    folds
)

lr_cv_results

,fold,accuracy,precision,recall,f1,auroc
0,1,0.538156,1.000000,0.475188,0.644241,0.830635
1,2,0.787744,0.738750,0.951691,0.831809,0.868801
2,3,0.537415,1.000000,0.463057,0.632999,0.858901
3,4,0.563137,0.828859,0.558192,0.667117,0.599464
4,5,0.551648,0.536597,0.994068,0.696970,0.728045


In [75]:
print("Logistic Regression — GroupKFold Mean")
print("-------------------------------------")

print(
    lr_cv_results[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].mean()
)

Logistic Regression — GroupKFold Mean
-------------------------------------
accuracy     0.595620
precision    0.820841
recall       0.688439
f1           0.694627
auroc        0.777169
dtype: float64


In [76]:
xgb_cv = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_cv_results = evaluate_cv_model(
    xgb_cv,
    X,
    y,
    groups,
    folds
)

xgb_cv_results

,fold,accuracy,precision,recall,f1,auroc
0,1,0.905161,1.000000,0.892231,0.943046,1.000000
1,2,0.680284,0.785558,0.578100,0.666048,0.666339
2,3,0.630709,0.819962,0.732092,0.773540,0.441840
3,4,0.672574,0.858733,0.697175,0.769567,0.615602
4,5,0.518681,0.518681,1.000000,0.683068,0.584034


In [77]:
print("XGBoost — GroupKFold Mean")
print("-------------------------")

print(
    xgb_cv_results[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].mean()
)

XGBoost — GroupKFold Mean
-------------------------
accuracy     0.681482
precision    0.796587
recall       0.779920
f1           0.767054
auroc        0.661563
dtype: float64


In [78]:
rf_cv = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_cv_results = evaluate_cv_model(
    rf_cv,
    X,
    y,
    groups,
    folds
)

rf_cv_results

,fold,accuracy,precision,recall,f1,auroc
0,1,0.831495,0.877045,0.940351,0.907596,0.802969
1,2,0.791741,0.831760,0.780193,0.805152,0.629597
2,3,0.746356,0.843493,0.866328,0.854758,0.557000
3,4,0.659725,0.855824,0.680791,0.758339,0.674301
4,5,0.518681,0.518681,1.000000,0.683068,0.628254


In [79]:
print("Random Forest — GroupKFold Mean")
print("--------------------------------")

print(
    rf_cv_results[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].mean()
)

Random Forest — GroupKFold Mean
--------------------------------
accuracy     0.709600
precision    0.785361
recall       0.853533
f1           0.801782
auroc        0.658424
dtype: float64


In [80]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

rf_window_oof_full = np.zeros(len(ml_dataset))

for fold_num, (train_idx, val_idx) in enumerate(folds, start=1):

    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]

    y_train_fold = y.iloc[train_idx]

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    rf_model.fit(X_train_fold, y_train_fold)

    rf_window_oof_full[val_idx] = rf_model.predict_proba(
        X_val_fold
    )[:, 1]

ml_dataset["rf_probability"] = rf_window_oof_full

print("RF probability added.")
print(ml_dataset[
    ["patient_id", "window_id", "risk_label", "rf_probability"]
].head())

print("\nShape:", ml_dataset.shape)
print("RF probability range:",
      ml_dataset["rf_probability"].min(),
      ml_dataset["rf_probability"].max())

RF probability added.
  patient_id  window_id  risk_label  rf_probability
0      chf01          0           1        0.973333
1      chf01          1           1        0.963333
2      chf01          2           1        0.970000
3      chf01          3           1        0.983333
4      chf01          4           1        0.983333

Shape: (11109, 18)
RF probability range: 0.01 1.0


In [81]:
print(
    ml_dataset[
        ["patient_id", "window_id", "risk_label", "rf_probability"]
    ].head(20)
)

print("\nPatients:", ml_dataset["patient_id"].nunique())
print("Windows:", len(ml_dataset))

print(
    "\nProbability missing values:",
    ml_dataset["rf_probability"].isna().sum()
)

print(
    "\nProbabilities per patient:"
)

print(
    ml_dataset.groupby("patient_id")["rf_probability"]
    .count()
    .describe()
)

   patient_id  window_id  risk_label  rf_probability
0       chf01          0           1        0.973333
1       chf01          1           1        0.963333
2       chf01          2           1        0.970000
3       chf01          3           1        0.983333
4       chf01          4           1        0.983333
5       chf01          5           1        0.983333
6       chf01          6           1        0.986667
7       chf01          7           1        0.986667
8       chf01          8           1        0.966667
9       chf01          9           1        0.966667
10      chf01         10           1        0.953333
11      chf01         11           1        0.960000
12      chf01         12           1        0.976667
13      chf01         13           1        0.966667
14      chf01         14           1        0.980000
15      chf01         15           1        0.960000
16      chf01         16           1        0.976667
17      chf01         17           1        0.

In [82]:
personal_features = [
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_hf"
]

ml_dataset = ml_dataset.sort_values(
    ["patient_id", "window_id"]
).reset_index(drop=True)

for feature in personal_features:

    ml_dataset[f"{feature}_robust_baseline"] = (
        ml_dataset
        .groupby("patient_id")[feature]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                window=30,
                min_periods=5
            )
            .median()
        )
    )

    ml_dataset[f"{feature}_robust_mad"] = (
        ml_dataset
        .groupby("patient_id")[feature]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                window=30,
                min_periods=5
            )
            .apply(
                lambda z: np.median(
                    np.abs(z - np.median(z))
                ),
                raw=True
            )
        )
    )

print("Robust patient-specific baselines created.")

print(
    ml_dataset[
        [
            "patient_id",
            "window_id",
            "sdnn",
            "sdnn_robust_baseline",
            "sdnn_robust_mad"
        ]
    ].head(40)
)

Robust patient-specific baselines created.
   patient_id  window_id       sdnn  sdnn_robust_baseline  sdnn_robust_mad
0       chf01          0  38.977235                   NaN              NaN
1       chf01          1  24.194927                   NaN              NaN
2       chf01          2  40.970951                   NaN              NaN
3       chf01          3  33.713954                   NaN              NaN
4       chf01          4  30.488855                   NaN              NaN
5       chf01          5  24.080741             33.713954              NaN
6       chf01          6  44.868778             32.101404              NaN
7       chf01          7  81.308888             33.713954              NaN
8       chf01          8  19.686808             36.345595              NaN
9       chf01          9  28.819065             33.713954              NaN
10      chf01         10  30.037754             32.101404              NaN
11      chf01         11  21.690820             30.488855

In [83]:
personal_features = ["sdnn", "rmssd", "pnn50", "lf_hf"]

for feature in personal_features:
    baseline_col = f"{feature}_robust_baseline"
    mad_col = f"{feature}_robust_mad"
    z_col = f"{feature}_robust_z"

    ml_dataset[z_col] = (
        (ml_dataset[feature] - ml_dataset[baseline_col])
        / (1.4826 * ml_dataset[mad_col] + 1e-6)
    )

    ml_dataset[z_col] = ml_dataset[z_col].clip(-10, 10)

print("Robust personalized z-scores created.")

print(
    ml_dataset[
        ["patient_id", "window_id"] +
        [f"{f}_robust_z" for f in personal_features]
    ].head(40)
)

Robust personalized z-scores created.
   patient_id  window_id  sdnn_robust_z  rmssd_robust_z  pnn50_robust_z  \
0       chf01          0            NaN             NaN             NaN   
1       chf01          1            NaN             NaN             NaN   
2       chf01          2            NaN             NaN             NaN   
3       chf01          3            NaN             NaN             NaN   
4       chf01          4            NaN             NaN             NaN   
5       chf01          5            NaN             NaN             NaN   
6       chf01          6            NaN             NaN             NaN   
7       chf01          7            NaN             NaN             NaN   
8       chf01          8            NaN             NaN             NaN   
9       chf01          9            NaN             NaN             NaN   
10      chf01         10            NaN             NaN             NaN   
11      chf01         11            NaN             NaN       

In [84]:
robust_z_cols = [
    "sdnn_robust_z",
    "rmssd_robust_z",
    "pnn50_robust_z",
    "lf_hf_robust_z"
]

ml_dataset["robust_drift_score"] = (
    ml_dataset[robust_z_cols]
    .abs()
    .mean(axis=1)
)

print("Robust drift score created.")

print(
    ml_dataset[
        ["patient_id", "window_id", "robust_drift_score"]
    ].head(40)
)

Robust drift score created.
   patient_id  window_id  robust_drift_score
0       chf01          0                 NaN
1       chf01          1                 NaN
2       chf01          2                 NaN
3       chf01          3                 NaN
4       chf01          4                 NaN
5       chf01          5                 NaN
6       chf01          6                 NaN
7       chf01          7                 NaN
8       chf01          8                 NaN
9       chf01          9                 NaN
10      chf01         10                 NaN
11      chf01         11                 NaN
12      chf01         12                 NaN
13      chf01         13                 NaN
14      chf01         14                 NaN
15      chf01         15                 NaN
16      chf01         16                 NaN
17      chf01         17                 NaN
18      chf01         18                 NaN
19      chf01         19                 NaN
20      chf01         20   

In [85]:
DRIFT_THRESHOLD = 2.0
PERSISTENCE = 3

ml_dataset["robust_drift_candidate"] = (
    ml_dataset["robust_drift_score"] >= DRIFT_THRESHOLD
).astype(int)

ml_dataset["robust_drift_persistence"] = (
    ml_dataset
    .groupby("patient_id")["robust_drift_candidate"]
    .transform(
        lambda x: x.rolling(PERSISTENCE, min_periods=1).sum()
    )
)

ml_dataset["robust_drift_flag"] = (
    ml_dataset["robust_drift_persistence"] >= PERSISTENCE
).astype(int)

print(
    ml_dataset["robust_drift_flag"].value_counts()
)

print(
    ml_dataset[
        [
            "patient_id",
            "window_id",
            "robust_drift_score",
            "robust_drift_persistence",
            "robust_drift_flag"
        ]
    ].head(40)
)

robust_drift_flag
0    10817
1      292
Name: count, dtype: int64
   patient_id  window_id  robust_drift_score  robust_drift_persistence  \
0       chf01          0                 NaN                       0.0   
1       chf01          1                 NaN                       0.0   
2       chf01          2                 NaN                       0.0   
3       chf01          3                 NaN                       0.0   
4       chf01          4                 NaN                       0.0   
5       chf01          5                 NaN                       0.0   
6       chf01          6                 NaN                       0.0   
7       chf01          7                 NaN                       0.0   
8       chf01          8                 NaN                       0.0   
9       chf01          9                 NaN                       0.0   
10      chf01         10                 NaN                       0.0   
11      chf01         11                 NaN  

In [86]:
patient_personalized = (
    ml_dataset
    .sort_values(["patient_id", "window_id"])
    .groupby("patient_id")
    .agg(
        risk_label=("risk_label", "first"),
        rf_probability=("rf_probability", "last"),
        mean_robust_drift=("robust_drift_score", "mean"),
        current_robust_drift=("robust_drift_score", "last"),
        max_robust_drift=("robust_drift_score", "max"),
        drift_windows=("robust_drift_flag", "sum"),
        current_drift_flag=("robust_drift_flag", "last"),
        history_count=("window_id", "count")
    )
    .reset_index()
)

print(patient_personalized.head())
print()
print("Shape:", patient_personalized.shape)

  patient_id  risk_label  rf_probability  mean_robust_drift  \
0      chf01           1        0.973333           1.132653   
1      chf02           1        0.946667           1.276555   
2      chf03           1        0.673333           1.057777   
3      chf04           1        0.450000           1.373736   
4      chf05           1        0.673333           1.071091   

   current_robust_drift  max_robust_drift  drift_windows  current_drift_flag  \
0              0.504898          6.243779              9                   0   
1              0.918036          4.897394             15                   0   
2              0.893690          6.156222              4                   0   
3              1.362044          8.277350             13                   0   
4              2.173413          5.434263              1                   0   

   history_count  
0            240  
1            230  
2            240  
3            240  
4            238  

Shape: (44, 9)


In [87]:
patient_personalized["personalized_probability"] = (
    0.80 * patient_personalized["rf_probability"]
    + 0.20 * (
        patient_personalized["current_robust_drift"]
        .clip(0, 3) / 3
    )
)

patient_personalized["personalized_probability"] = (
    patient_personalized["personalized_probability"]
    .clip(0, 1)
)

print(
    patient_personalized[
        [
            "patient_id",
            "risk_label",
            "rf_probability",
            "current_robust_drift",
            "personalized_probability"
        ]
    ].head(20)
)

   patient_id  risk_label  rf_probability  current_robust_drift  \
0       chf01           1        0.973333              0.504898   
1       chf02           1        0.946667              0.918036   
2       chf03           1        0.673333              0.893690   
3       chf04           1        0.450000              1.362044   
4       chf05           1        0.673333              2.173413   
5       chf06           1        0.236667              0.475817   
6       chf07           1        0.573333              2.265605   
7       chf08           1        0.480000              0.439315   
8       chf09           1        0.793333              1.211968   
9       chf10           1        0.743333              1.161385   
10      chf11           1        0.950000              0.562896   
11      chf12           1        0.830000              1.036846   
12      chf13           1        0.793333              2.487107   
13      chf14           1        0.876667              3.77610

In [88]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_true = patient_personalized["risk_label"]

rf_prob = patient_personalized["rf_probability"]
personalized_prob = patient_personalized["personalized_probability"]

rf_pred = (rf_prob >= 0.5).astype(int)
personalized_pred = (personalized_prob >= 0.5).astype(int)

comparison = pd.DataFrame({
    "RF Baseline": [
        accuracy_score(y_true, rf_pred),
        precision_score(y_true, rf_pred, zero_division=0),
        recall_score(y_true, rf_pred, zero_division=0),
        f1_score(y_true, rf_pred, zero_division=0),
        roc_auc_score(y_true, rf_prob)
    ],
    "Personalized Model": [
        accuracy_score(y_true, personalized_pred),
        precision_score(y_true, personalized_pred, zero_division=0),
        recall_score(y_true, personalized_pred, zero_division=0),
        f1_score(y_true, personalized_pred, zero_division=0),
        roc_auc_score(y_true, personalized_prob)
    ]
}, index=[
    "accuracy",
    "precision",
    "recall",
    "f1",
    "auroc"
])

print(comparison)

           RF Baseline  Personalized Model
accuracy      0.681818            0.681818
precision     0.764706            0.764706
recall        0.812500            0.812500
f1            0.787879            0.787879
auroc         0.566406            0.552083


In [89]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [90]:
print("ml_dataset:", ml_dataset.shape)
print("X:", X.shape)
print("y:", y.shape)
print("groups:", groups.shape)
print("folds:", len(folds))

print("\nPatients:", ml_dataset["patient_id"].nunique())

ml_dataset: (11109, 34)
X: (11109, 11)
y: (11109,)
groups: (11109,)
folds: 5

Patients: 44


In [91]:
def create_patient_personal_features(df):
    
    df = df.sort_values(
        ["patient_id", "window_id"]
    ).copy()
    
    personal_features = [
        "sdnn",
        "rmssd",
        "pnn50",
        "lf_hf"
    ]
    
    results = []
    
    for patient_id, patient_df in df.groupby("patient_id"):
        
        patient_df = patient_df.sort_values("window_id").copy()
        
        for feature in personal_features:
            
            previous = patient_df[feature].shift(1)
            
            rolling_mean = previous.rolling(
                window=30,
                min_periods=5
            ).mean()
            
            rolling_std = previous.rolling(
                window=30,
                min_periods=5
            ).std()
            
            patient_df[f"{feature}_z"] = (
                patient_df[feature] - rolling_mean
            ) / (rolling_std + 1e-6)
        
        z_cols = [
            "sdnn_z",
            "rmssd_z",
            "pnn50_z",
            "lf_hf_z"
        ]
        
        patient_df["personal_drift_score"] = (
            patient_df[z_cols]
            .abs()
            .mean(axis=1)
        )
        
        patient_df["drift_flag"] = (
            patient_df["personal_drift_score"] >= 2.0
        ).astype(int)
        
        results.append(patient_df)
    
    result = pd.concat(results, ignore_index=True)
    
    return result

In [92]:
personalized_data = create_patient_personal_features(
    ml_dataset
)

print(
    personalized_data[
        [
            "patient_id",
            "window_id",
            "personal_drift_score",
            "drift_flag"
        ]
    ].head(20)
)

   patient_id  window_id  personal_drift_score  drift_flag
0       chf01          0                   NaN           0
1       chf01          1                   NaN           0
2       chf01          2                   NaN           0
3       chf01          3                   NaN           0
4       chf01          4                   NaN           0
5       chf01          5              1.099019           0
6       chf01          6              1.036639           0
7       chf01          7              2.111831           1
8       chf01          8              0.800840           0
9       chf01          9              0.825753           0
10      chf01         10              0.658025           0
11      chf01         11              0.688162           0
12      chf01         12              0.522517           0
13      chf01         13              0.912968           0
14      chf01         14              0.193154           0
15      chf01         15              0.729115          

In [93]:
personalized_data["personal_drift_score"] = (
    personalized_data["personal_drift_score"]
    .fillna(0)
)

personalized_data["drift_flag"] = (
    personalized_data["drift_flag"]
    .fillna(0)
    .astype(int)
)

print(
    personalized_data[
        [
            "personal_drift_score",
            "drift_flag"
        ]
    ].describe()
)

       personal_drift_score    drift_flag
count          11109.000000  11109.000000
mean               0.885493      0.051040
std                1.127655      0.220089
min                0.000000      0.000000
25%                0.473611      0.000000
50%                0.713576      0.000000
75%                1.067397      0.000000
max               91.780886      1.000000


In [94]:
patient_personal_summary = (
    personalized_data
    .groupby("patient_id")
    .agg(
        risk_label=("risk_label", "first"),
        
        mean_personal_drift=(
            "personal_drift_score",
            "mean"
        ),
        
        max_personal_drift=(
            "personal_drift_score",
            "max"
        ),
        
        drift_windows=(
            "drift_flag",
            "sum"
        ),
        
        total_windows=(
            "drift_flag",
            "count"
        )
    )
    .reset_index()
)

patient_personal_summary["drift_rate"] = (
    patient_personal_summary["drift_windows"]
    /
    patient_personal_summary["total_windows"]
)

print(patient_personal_summary.head())
print()
print("Shape:", patient_personal_summary.shape)

  patient_id  risk_label  mean_personal_drift  max_personal_drift  \
0      chf01           1             0.880257            6.404265   
1      chf02           1             1.055789            3.267557   
2      chf03           1             0.801786            3.494529   
3      chf04           1             0.942942            5.953177   
4      chf05           1             0.828081            4.572772   

   drift_windows  total_windows  drift_rate  
0             13            240    0.054167  
1             24            230    0.104348  
2             12            240    0.050000  
3             21            240    0.087500  
4             11            238    0.046218  

Shape: (44, 7)


In [95]:
print(
    patient_personal_summary["risk_label"]
    .value_counts()
)

risk_label
1    32
0    12
Name: count, dtype: int64


In [96]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import numpy as np
import pandas as pd

patient_fold_results = []

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(
    gkf.split(
        ml_dataset,
        ml_dataset["risk_label"],
        groups=ml_dataset["patient_id"]
    ),
    start=1
):

    val_df = ml_dataset.iloc[val_idx].copy()

    patient_fold = (
        val_df
        .groupby("patient_id")
        .agg(
            risk_label=("risk_label", "first"),
            rf_probability=("rf_probability", "mean")
        )
        .reset_index()
    )

    patient_fold["rf_prediction"] = (
        patient_fold["rf_probability"] >= 0.5
    ).astype(int)

    accuracy = accuracy_score(
        patient_fold["risk_label"],
        patient_fold["rf_prediction"]
    )

    precision = precision_score(
        patient_fold["risk_label"],
        patient_fold["rf_prediction"],
        zero_division=0
    )

    recall = recall_score(
        patient_fold["risk_label"],
        patient_fold["rf_prediction"],
        zero_division=0
    )

    f1 = f1_score(
        patient_fold["risk_label"],
        patient_fold["rf_prediction"],
        zero_division=0
    )

    try:
        auroc = roc_auc_score(
            patient_fold["risk_label"],
            patient_fold["rf_probability"]
        )
    except ValueError:
        auroc = np.nan

    patient_fold_results.append({
        "fold": fold,
        "patients": len(patient_fold),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auroc": auroc
    })

    print(
        f"Fold {fold}: "
        f"Patients={len(patient_fold)}, "
        f"Accuracy={accuracy:.4f}, "
        f"F1={f1:.4f}, "
        f"AUROC={auroc:.4f}"
    )

patient_fold_results_df = pd.DataFrame(patient_fold_results)

print("\nRF — Patient-Level GroupKFold")
print("--------------------------------")

display(patient_fold_results_df)

print("\nMean ± Standard Deviation")

display(
    patient_fold_results_df[
        ["accuracy", "precision", "recall", "f1", "auroc"]
    ].agg(["mean", "std"])
)

Fold 1: Patients=9, Accuracy=0.7778, F1=0.8750, AUROC=0.7500
Fold 2: Patients=9, Accuracy=0.7778, F1=0.8000, AUROC=0.6000
Fold 3: Patients=8, Accuracy=0.7500, F1=0.8571, AUROC=0.5714
Fold 4: Patients=9, Accuracy=0.6667, F1=0.7692, AUROC=0.6429
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.6500

RF — Patient-Level GroupKFold
--------------------------------


,fold,patients,accuracy,precision,recall,f1,auroc
0,1,9,0.777778,0.875000,0.875000,0.875000,0.750000
1,2,9,0.777778,0.800000,0.800000,0.800000,0.600000
2,3,8,0.750000,0.857143,0.857143,0.857143,0.571429
3,4,9,0.666667,0.833333,0.714286,0.769231,0.642857
4,5,9,0.555556,0.555556,1.000000,0.714286,0.650000



Mean ± Standard Deviation


,accuracy,precision,recall,f1,auroc
mean,0.705556,0.784206,0.849286,0.803132,0.642857
std,0.095420,0.130870,0.104990,0.065450,0.067951


In [97]:
#Patient-Level GroupKFold Evaluation

In [98]:
from sklearn.base import clone

def evaluate_patient_level_cv(model, X, y, ml_dataset, folds):

    fold_results = []
    all_patient_predictions = []

    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):

        X_train_cv = X.iloc[train_idx]
        X_val_cv = X.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model_fold = clone(model)

        model_fold.fit(
            X_train_cv,
            y_train_cv
        )

        y_prob_cv = model_fold.predict_proba(
            X_val_cv
        )[:, 1]

        validation_data = ml_dataset.iloc[
            val_idx
        ][
            ["patient_id", "risk_label"]
        ].copy()

        validation_data["predicted_probability"] = y_prob_cv

        patient_data = (
            validation_data
            .groupby(["patient_id", "risk_label"])
            .agg(
                mean_probability=(
                    "predicted_probability",
                    "mean"
                ),
                window_count=(
                    "predicted_probability",
                    "count"
                )
            )
            .reset_index()
        )

        patient_data["predicted_class"] = (
            patient_data["mean_probability"] >= 0.5
        ).astype(int)

        y_true_patient = patient_data["risk_label"]
        y_pred_patient = patient_data["predicted_class"]
        y_prob_patient = patient_data["mean_probability"]

        fold_result = {
            "fold": fold_number,
            "patients": len(patient_data),
            "accuracy": accuracy_score(
                y_true_patient,
                y_pred_patient
            ),
            "precision": precision_score(
                y_true_patient,
                y_pred_patient,
                zero_division=0
            ),
            "recall": recall_score(
                y_true_patient,
                y_pred_patient,
                zero_division=0
            ),
            "f1": f1_score(
                y_true_patient,
                y_pred_patient,
                zero_division=0
            ),
            "auroc": roc_auc_score(
                y_true_patient,
                y_prob_patient
            )
        }

        fold_results.append(fold_result)

        patient_data["fold"] = fold_number
        all_patient_predictions.append(patient_data)

        print(
            f"Fold {fold_number}: "
            f"Patients={len(patient_data)}, "
            f"Accuracy={fold_result['accuracy']:.4f}, "
            f"F1={fold_result['f1']:.4f}, "
            f"AUROC={fold_result['auroc']:.4f}"
        )

    fold_results_df = pd.DataFrame(fold_results)

    patient_predictions_df = pd.concat(
        all_patient_predictions,
        ignore_index=True
    )

    summary = pd.DataFrame({
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "AUROC"
        ],
        "Mean": [
            fold_results_df["accuracy"].mean(),
            fold_results_df["precision"].mean(),
            fold_results_df["recall"].mean(),
            fold_results_df["f1"].mean(),
            fold_results_df["auroc"].mean()
        ],
        "Std": [
            fold_results_df["accuracy"].std(),
            fold_results_df["precision"].std(),
            fold_results_df["recall"].std(),
            fold_results_df["f1"].std(),
            fold_results_df["auroc"].std()
        ]
    })

    return (
        fold_results_df,
        summary,
        patient_predictions_df
    )

In [99]:
lr_patient_results, lr_patient_summary, lr_patient_predictions = \
    evaluate_patient_level_cv(
        lr_cv,
        X,
        y,
        ml_dataset,
        folds
    )

print("\nLogistic Regression — Patient-Level GroupKFold")
print("-----------------------------------------------")
print(lr_patient_summary)

Fold 1: Patients=9, Accuracy=0.5556, F1=0.6667, AUROC=0.8750
Fold 2: Patients=9, Accuracy=0.7778, F1=0.8333, AUROC=0.9000
Fold 3: Patients=8, Accuracy=0.6250, F1=0.7273, AUROC=0.8571
Fold 4: Patients=9, Accuracy=0.5556, F1=0.6667, AUROC=0.5714
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.8000

Logistic Regression — Patient-Level GroupKFold
-----------------------------------------------
      Metric      Mean       Std
0   Accuracy  0.613889  0.096425
1  Precision  0.813968  0.191131
2     Recall  0.728571  0.249489
3         F1  0.721645  0.068201
4      AUROC  0.800714  0.133354


In [100]:
xgb_patient_results, xgb_patient_summary, xgb_patient_predictions = \
    evaluate_patient_level_cv(
        xgb_cv,
        X,
        y,
        ml_dataset,
        folds
    )

print("\nXGBoost — Patient-Level GroupKFold")
print("----------------------------------")
print(xgb_patient_summary)

Fold 1: Patients=9, Accuracy=0.8889, F1=0.9333, AUROC=1.0000
Fold 2: Patients=9, Accuracy=0.6667, F1=0.6667, AUROC=0.6000
Fold 3: Patients=8, Accuracy=0.6250, F1=0.7692, AUROC=0.4286
Fold 4: Patients=9, Accuracy=0.6667, F1=0.7692, AUROC=0.5000
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.6000

XGBoost — Patient-Level GroupKFold
----------------------------------
      Metric      Mean       Std
0   Accuracy  0.680556  0.125000
1  Precision  0.794444  0.161494
2     Recall  0.780714  0.156891
3         F1  0.770549  0.100573
4      AUROC  0.625714  0.221406


In [101]:
rf_patient_results, rf_patient_summary, rf_patient_predictions = \
    evaluate_patient_level_cv(
        rf_cv,
        X,
        y,
        ml_dataset,
        folds
    )

print("\nRandom Forest — Patient-Level GroupKFold")
print("-----------------------------------------")
print(rf_patient_summary)

Fold 1: Patients=9, Accuracy=0.7778, F1=0.8750, AUROC=0.7500
Fold 2: Patients=9, Accuracy=0.7778, F1=0.8000, AUROC=0.6000
Fold 3: Patients=8, Accuracy=0.7500, F1=0.8571, AUROC=0.5714
Fold 4: Patients=9, Accuracy=0.6667, F1=0.7692, AUROC=0.6429
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.6500

Random Forest — Patient-Level GroupKFold
-----------------------------------------
      Metric      Mean       Std
0   Accuracy  0.705556  0.095420
1  Precision  0.784206  0.130870
2     Recall  0.849286  0.104990
3         F1  0.803132  0.065450
4      AUROC  0.642857  0.067951


In [102]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate_patient_level_cv(
    model,
    X,
    y,
    ml_dataset,
    folds
):

    fold_results = []
    all_patient_predictions = []

    for fold_number, (train_idx, val_idx) in enumerate(
        folds,
        start=1
    ):

        X_train_cv = X.iloc[train_idx]
        X_val_cv = X.iloc[val_idx]

        y_train_cv = y.iloc[train_idx]
        y_val_cv = y.iloc[val_idx]

        model.fit(
            X_train_cv,
            y_train_cv
        )

        y_prob_cv = model.predict_proba(
            X_val_cv
        )[:, 1]

        validation_patients = (
            ml_dataset.iloc[val_idx]["patient_id"]
            .values
        )

        patient_predictions = pd.DataFrame({
            "patient_id": validation_patients,
            "y_true": y_val_cv.values,
            "y_prob": y_prob_cv
        })

        patient_predictions = (
            patient_predictions
            .groupby("patient_id")
            .agg(
                y_true=("y_true", "first"),
                y_prob=("y_prob", "mean")
            )
            .reset_index()
        )

        patient_predictions["y_pred"] = (
            patient_predictions["y_prob"] >= 0.5
        ).astype(int)

        result = {
            "fold": fold_number,
            "patients": len(patient_predictions),
            "accuracy": accuracy_score(
                patient_predictions["y_true"],
                patient_predictions["y_pred"]
            ),
            "precision": precision_score(
                patient_predictions["y_true"],
                patient_predictions["y_pred"],
                zero_division=0
            ),
            "recall": recall_score(
                patient_predictions["y_true"],
                patient_predictions["y_pred"],
                zero_division=0
            ),
            "f1": f1_score(
                patient_predictions["y_true"],
                patient_predictions["y_pred"],
                zero_division=0
            ),
            "auroc": roc_auc_score(
                patient_predictions["y_true"],
                patient_predictions["y_prob"]
            )
        }

        fold_results.append(result)

        patient_predictions["fold"] = fold_number

        all_patient_predictions.append(
            patient_predictions
        )

        print(
            f"Fold {fold_number}: "
            f"Patients={result['patients']}, "
            f"Accuracy={result['accuracy']:.4f}, "
            f"F1={result['f1']:.4f}, "
            f"AUROC={result['auroc']:.4f}"
        )

    return (
        pd.DataFrame(fold_results),
        pd.concat(
            all_patient_predictions,
            ignore_index=True
        )
    )

In [103]:
lr_patient_results, lr_patient_predictions = \
    evaluate_patient_level_cv(
        lr_cv,
        X,
        y,
        ml_dataset,
        folds
    )

print("\nLogistic Regression — Patient-Level GroupKFold")
print("-----------------------------------------------")

print(
    lr_patient_results[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auroc"
        ]
    ].agg(["mean", "std"])
)

Fold 1: Patients=9, Accuracy=0.5556, F1=0.6667, AUROC=0.8750
Fold 2: Patients=9, Accuracy=0.7778, F1=0.8333, AUROC=0.9000
Fold 3: Patients=8, Accuracy=0.6250, F1=0.7273, AUROC=0.8571
Fold 4: Patients=9, Accuracy=0.5556, F1=0.6667, AUROC=0.5714
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.8000

Logistic Regression — Patient-Level GroupKFold
-----------------------------------------------
      accuracy  precision    recall        f1     auroc
mean  0.613889   0.813968  0.728571  0.721645  0.800714
std   0.096425   0.191131  0.249489  0.068201  0.133354


In [104]:
rf_patient_results, rf_patient_predictions = \
    evaluate_patient_level_cv(
        rf_cv,
        X,
        y,
        ml_dataset,
        folds
    )

print("\nRandom Forest — Patient-Level GroupKFold")
print("-----------------------------------------")

print(
    rf_patient_results[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auroc"
        ]
    ].agg(["mean", "std"])
)

Fold 1: Patients=9, Accuracy=0.7778, F1=0.8750, AUROC=0.7500
Fold 2: Patients=9, Accuracy=0.7778, F1=0.8000, AUROC=0.6000
Fold 3: Patients=8, Accuracy=0.7500, F1=0.8571, AUROC=0.5714
Fold 4: Patients=9, Accuracy=0.6667, F1=0.7692, AUROC=0.6429
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.6500

Random Forest — Patient-Level GroupKFold
-----------------------------------------
      accuracy  precision    recall        f1     auroc
mean  0.705556   0.784206  0.849286  0.803132  0.642857
std   0.095420   0.130870  0.104990  0.065450  0.067951


In [105]:
xgb_patient_results, xgb_patient_predictions = \
    evaluate_patient_level_cv(
        xgb_cv,
        X,
        y,
        ml_dataset,
        folds
    )

print("\nXGBoost — Patient-Level GroupKFold")
print("----------------------------------")

print(
    xgb_patient_results[
        [
            "accuracy",
            "precision",
            "recall",
            "f1",
            "auroc"
        ]
    ].agg(["mean", "std"])
)

Fold 1: Patients=9, Accuracy=0.8889, F1=0.9333, AUROC=1.0000
Fold 2: Patients=9, Accuracy=0.6667, F1=0.6667, AUROC=0.6000
Fold 3: Patients=8, Accuracy=0.6250, F1=0.7692, AUROC=0.4286
Fold 4: Patients=9, Accuracy=0.6667, F1=0.7692, AUROC=0.5000
Fold 5: Patients=9, Accuracy=0.5556, F1=0.7143, AUROC=0.6000

XGBoost — Patient-Level GroupKFold
----------------------------------
      accuracy  precision    recall        f1     auroc
mean  0.680556   0.794444  0.780714  0.770549  0.625714
std   0.125000   0.161494  0.156891  0.100573  0.221406


In [106]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

rf_oof = np.zeros(len(ml_dataset))

for fold_idx, (train_idx, val_idx) in enumerate(folds, start=1):

    X_train = X.iloc[train_idx]
    X_val = X.iloc[val_idx]

    y_train = y.iloc[train_idx]

    rf_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    )

    rf_model.fit(X_train, y_train)

    rf_oof[val_idx] = rf_model.predict_proba(X_val)[:, 1]

    print(
        f"Fold {fold_idx}: "
        f"Train={len(train_idx)}, "
        f"Validation={len(val_idx)}"
    )

print("\nOOF probability generation complete.")
print("Number of predictions:", len(rf_oof))

Fold 1: Train=8842, Validation=2267
Fold 2: Train=8857, Validation=2252
Fold 3: Train=9051, Validation=2058
Fold 4: Train=8852, Validation=2257
Fold 5: Train=8834, Validation=2275

OOF probability generation complete.
Number of predictions: 11109


In [107]:
ml_dataset["rf_oof_probability"] = rf_oof

print(
    ml_dataset[
        [
            "patient_id",
            "window_id",
            "risk_label",
            "rf_oof_probability"
        ]
    ].head(20)
)

   patient_id  window_id  risk_label  rf_oof_probability
0       chf01          0           1            0.973333
1       chf01          1           1            0.963333
2       chf01          2           1            0.970000
3       chf01          3           1            0.983333
4       chf01          4           1            0.983333
5       chf01          5           1            0.983333
6       chf01          6           1            0.986667
7       chf01          7           1            0.986667
8       chf01          8           1            0.966667
9       chf01          9           1            0.966667
10      chf01         10           1            0.953333
11      chf01         11           1            0.960000
12      chf01         12           1            0.976667
13      chf01         13           1            0.966667
14      chf01         14           1            0.980000
15      chf01         15           1            0.960000
16      chf01         16       

In [108]:
personal_features = [
    "sdnn",
    "rmssd",
    "pnn50",
    "lf_hf"
]

df_personal = ml_dataset[
    [
        "patient_id",
        "window_id",
        "risk_label",
        "rf_oof_probability"
    ] + personal_features
].copy()

df_personal = df_personal.sort_values(
    ["patient_id", "window_id"]
).reset_index(drop=True)

for feature in personal_features:

    df_personal[f"{feature}_baseline"] = (
        df_personal
        .groupby("patient_id")[feature]
        .transform(
            lambda x: x.shift(1).rolling(
                window=20,
                min_periods=5
            ).mean()
        )
    )

    df_personal[f"{feature}_baseline_std"] = (
        df_personal
        .groupby("patient_id")[feature]
        .transform(
            lambda x: x.shift(1).rolling(
                window=20,
                min_periods=5
            ).std()
        )
    )

    df_personal[f"{feature}_z"] = (
        (
            df_personal[feature]
            - df_personal[f"{feature}_baseline"]
        )
        /
        (
            df_personal[f"{feature}_baseline_std"]
            + 1e-6
        )
    )

print(df_personal.head(25))

   patient_id  window_id  risk_label  rf_oof_probability       sdnn  \
0       chf01          0           1            0.973333  38.977235   
1       chf01          1           1            0.963333  24.194927   
2       chf01          2           1            0.970000  40.970951   
3       chf01          3           1            0.983333  33.713954   
4       chf01          4           1            0.983333  30.488855   
5       chf01          5           1            0.983333  24.080741   
6       chf01          6           1            0.986667  44.868778   
7       chf01          7           1            0.986667  81.308888   
8       chf01          8           1            0.966667  19.686808   
9       chf01          9           1            0.966667  28.819065   
10      chf01         10           1            0.953333  30.037754   
11      chf01         11           1            0.960000  21.690820   
12      chf01         12           1            0.976667  51.222382   
13    

In [109]:
z_columns = [
    f"{feature}_z"
    for feature in personal_features
]

df_personal["personal_z_score"] = (
    df_personal[z_columns]
    .abs()
    .mean(axis=1)
)

df_personal["history_count"] = (
    df_personal
    .groupby("patient_id")
    .cumcount()
)

df_personal["baseline_confidence"] = np.where(
    df_personal["history_count"] >= 5,
    1,
    0
)

print(
    df_personal[
        [
            "patient_id",
            "window_id",
            "rf_oof_probability",
            "personal_z_score",
            "history_count",
            "baseline_confidence"
        ]
    ].head(30)
)

   patient_id  window_id  rf_oof_probability  personal_z_score  history_count  \
0       chf01          0            0.973333               NaN              0   
1       chf01          1            0.963333               NaN              1   
2       chf01          2            0.970000               NaN              2   
3       chf01          3            0.983333               NaN              3   
4       chf01          4            0.983333               NaN              4   
5       chf01          5            0.983333          1.099019              5   
6       chf01          6            0.986667          1.036639              6   
7       chf01          7            0.986667          2.111831              7   
8       chf01          8            0.966667          0.800840              8   
9       chf01          9            0.966667          0.825753              9   
10      chf01         10            0.953333          0.658025             10   
11      chf01         11    

In [114]:
print("Variables containing drift/patient/state:")
print([
    name for name in globals()
    if any(x in name.lower() for x in ["drift", "patient_state", "personal"])
])  
print("Current variables containing DataFrame objects:")
print([
    name for name, value in globals().items()
    if hasattr(value, "columns")
])

Variables containing drift/patient/state:
['personal_features', 'DRIFT_THRESHOLD', 'patient_personalized', 'personalized_prob', 'personalized_pred', 'create_patient_personal_features', 'personalized_data', 'patient_personal_summary', 'df_personal']
Current variables containing DataFrame objects:
['_', '__', '___', 'ml_dataset', 'X', 'patient_labels', 'test_patients', 'train_df', 'test_df', 'X_train', 'X_test', 'test_results', 'patient_results', 'rf_test_results', 'rf_patient_results', 'rf_importance', 'X_train_base', 'X_test_base', 'rf_base_results', 'rf_base_patient', 'xgb_test_results', 'xgb_patient_results', 'xgb_importance', 'baseline_results', '_63', 'patient_label_check', 'lr_cv_results', '_74', 'xgb_cv_results', '_76', 'rf_cv_results', '_78', 'X_train_fold', 'X_val_fold', 'patient_personalized', 'comparison', 'personalized_data', 'patient_personal_summary', 'val_df', 'patient_fold', 'patient_fold_results_df', 'lr_patient_results', 'lr_patient_summary', 'lr_patient_predictions', 

In [115]:
df_personal["drift_event"] = (
    (df_personal["personal_z_score"] >= 2.0)
    &
    (df_personal["baseline_confidence"] == 1)
).astype(int)

df_personal["drift_persistence"] = (
    df_personal
    .groupby("patient_id")["drift_event"]
    .transform(
        lambda x: x.rolling(
            window=5,
            min_periods=1
        ).sum()
    )
)

print(
    df_personal[
        [
            "patient_id",
            "window_id",
            "personal_z_score",
            "drift_event",
            "drift_persistence"
        ]
    ].head(30)
)

   patient_id  window_id  personal_z_score  drift_event  drift_persistence
0       chf01          0               NaN            0                0.0
1       chf01          1               NaN            0                0.0
2       chf01          2               NaN            0                0.0
3       chf01          3               NaN            0                0.0
4       chf01          4               NaN            0                0.0
5       chf01          5          1.099019            0                0.0
6       chf01          6          1.036639            0                0.0
7       chf01          7          2.111831            1                1.0
8       chf01          8          0.800840            0                1.0
9       chf01          9          0.825753            0                1.0
10      chf01         10          0.658025            0                1.0
11      chf01         11          0.688162            0                1.0
12      chf01         12 

In [116]:
df_personal["personalized_probability"] = (
    0.75 * df_personal["rf_oof_probability"]
    +
    0.15 * np.clip(
        df_personal["personal_z_score"] / 3.0,
        0,
        1
    )
    +
    0.10 * np.clip(
        df_personal["drift_persistence"] / 5.0,
        0,
        1
    )
)

df_personal["personalized_probability"] = (
    df_personal["personalized_probability"]
    .clip(0, 1)
)

print(
    df_personal[
        [
            "patient_id",
            "window_id",
            "rf_oof_probability",
            "personal_z_score",
            "drift_persistence",
            "personalized_probability"
        ]
    ].head(20)
)

   patient_id  window_id  rf_oof_probability  personal_z_score  \
0       chf01          0            0.973333               NaN   
1       chf01          1            0.963333               NaN   
2       chf01          2            0.970000               NaN   
3       chf01          3            0.983333               NaN   
4       chf01          4            0.983333               NaN   
5       chf01          5            0.983333          1.099019   
6       chf01          6            0.986667          1.036639   
7       chf01          7            0.986667          2.111831   
8       chf01          8            0.966667          0.800840   
9       chf01          9            0.966667          0.825753   
10      chf01         10            0.953333          0.658025   
11      chf01         11            0.960000          0.688162   
12      chf01         12            0.976667          0.522517   
13      chf01         13            0.966667          0.912968   
14      ch

In [117]:
patient_personalized = (
    df_personal
    .groupby("patient_id")
    .agg(
        risk_label=("risk_label", "first"),
        rf_probability=("rf_oof_probability", "mean"),
        personalized_probability=(
            "personalized_probability",
            "mean"
        ),
        mean_personal_z=(
            "personal_z_score",
            "mean"
        ),
        max_personal_z=(
            "personal_z_score",
            "max"
        ),
        max_drift_persistence=(
            "drift_persistence",
            "max"
        ),
        drift_windows=(
            "drift_event",
            "sum"
        ),
        history_count=(
            "history_count",
            "max"
        ),
        baseline_confidence=(
            "baseline_confidence",
            "max"
        )
    )
    .reset_index()
)

print(patient_personalized)

print("\nShape:", patient_personalized.shape)

   patient_id  risk_label  rf_probability  personalized_probability  \
0       chf01           1        0.973139                  0.779910   
1       chf02           1        0.958493                  0.780303   
2       chf03           1        0.708278                  0.580506   
3       chf04           1        0.385208                  0.346938   
4       chf05           1        0.719146                  0.587844   
5       chf06           1        0.240084                  0.234308   
6       chf07           1        0.666861                  0.553305   
7       chf08           1        0.529306                  0.449023   
8       chf09           1        0.801036                  0.653709   
9       chf10           1        0.759596                  0.618207   
10      chf11           1        0.993639                  0.795107   
11      chf12           1        0.849230                  0.687012   
12      chf13           1        0.776833                  0.637002   
13    

In [118]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_true = patient_personalized["risk_label"]

rf_prob_patient = (
    patient_personalized["rf_probability"]
)

personalized_prob_patient = (
    patient_personalized["personalized_probability"]
)

rf_pred_patient = (
    rf_prob_patient >= 0.5
).astype(int)

personalized_pred_patient = (
    personalized_prob_patient >= 0.5
).astype(int)

rf_metrics = {
    "accuracy": accuracy_score(
        y_true,
        rf_pred_patient
    ),
    "precision": precision_score(
        y_true,
        rf_pred_patient,
        zero_division=0
    ),
    "recall": recall_score(
        y_true,
        rf_pred_patient,
        zero_division=0
    ),
    "f1": f1_score(
        y_true,
        rf_pred_patient,
        zero_division=0
    ),
    "auroc": roc_auc_score(
        y_true,
        rf_prob_patient
    )
}

personalized_metrics = {
    "accuracy": accuracy_score(
        y_true,
        personalized_pred_patient
    ),
    "precision": precision_score(
        y_true,
        personalized_pred_patient,
        zero_division=0
    ),
    "recall": recall_score(
        y_true,
        personalized_pred_patient,
        zero_division=0
    ),
    "f1": f1_score(
        y_true,
        personalized_pred_patient,
        zero_division=0
    ),
    "auroc": roc_auc_score(
        y_true,
        personalized_prob_patient
    )
}

comparison = pd.DataFrame(
    [
        rf_metrics,
        personalized_metrics
    ],
    index=[
        "RF Baseline",
        "Personalized Model"
    ]
)

print(comparison)

                    accuracy  precision   recall        f1     auroc
RF Baseline         0.704545   0.771429  0.84375  0.805970  0.554688
Personalized Model  0.659091   0.757576  0.78125  0.769231  0.541667


In [119]:
check = pd.read_csv(
    "../data/processed/features/patient_rf_oof.csv"
)

print(check.shape)
print(check.head())

(44, 3)
  patient_id  risk_label  rf_probability
0      chf01           1        0.973139
1      chf02           1        0.958493
2      chf03           1        0.708278
3      chf04           1        0.385208
4      chf05           1        0.719146


In [120]:
ml_dataset["rf_oof_probability"] = rf_oof

In [121]:
print(
    ml_dataset[
        [
            "patient_id",
            "window_id",
            "risk_label",
            "rf_oof_probability"
        ]
    ].head(20)
)

   patient_id  window_id  risk_label  rf_oof_probability
0       chf01          0           1            0.973333
1       chf01          1           1            0.963333
2       chf01          2           1            0.970000
3       chf01          3           1            0.983333
4       chf01          4           1            0.983333
5       chf01          5           1            0.983333
6       chf01          6           1            0.986667
7       chf01          7           1            0.986667
8       chf01          8           1            0.966667
9       chf01          9           1            0.966667
10      chf01         10           1            0.953333
11      chf01         11           1            0.960000
12      chf01         12           1            0.976667
13      chf01         13           1            0.966667
14      chf01         14           1            0.980000
15      chf01         15           1            0.960000
16      chf01         16       

In [122]:
rf_oof_dataset = ml_dataset[
    [
        "patient_id",
        "window_id",
        "risk_label",
        "rf_oof_probability"
    ]
].copy()

rf_oof_dataset.to_csv(
    "../data/processed/features/rf_window_oof.csv",
    index=False
)

print("Saved RF window-level OOF predictions.")
print("Shape:", rf_oof_dataset.shape)
print(rf_oof_dataset.head())

Saved RF window-level OOF predictions.
Shape: (11109, 4)
  patient_id  window_id  risk_label  rf_oof_probability
0      chf01          0           1            0.973333
1      chf01          1           1            0.963333
2      chf01          2           1            0.970000
3      chf01          3           1            0.983333
4      chf01          4           1            0.983333


In [123]:
patient_oof = (
    ml_dataset
    .groupby("patient_id")
    .agg(
        risk_label=("risk_label", "first"),
        rf_probability=("rf_oof_probability", "mean")
    )
    .reset_index()
)

In [125]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

patient_oof["rf_prediction"] = (
    patient_oof["rf_probability"] >= 0.5
).astype(int)

print("RF Patient-Level OOF Evaluation")
print("--------------------------------")

print(
    "Accuracy:",
    accuracy_score(
        patient_oof["risk_label"],
        patient_oof["rf_prediction"]
    )
)

print(
    "Precision:",
    precision_score(
        patient_oof["risk_label"],
        patient_oof["rf_prediction"],
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        patient_oof["risk_label"],
        patient_oof["rf_prediction"],
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        patient_oof["risk_label"],
        patient_oof["rf_prediction"],
        zero_division=0
    )
)

print(
    "AUROC:",
    roc_auc_score(
        patient_oof["risk_label"],
        patient_oof["rf_probability"]
    )
)

RF Patient-Level OOF Evaluation
--------------------------------
Accuracy: 0.7045454545454546
Precision: 0.7714285714285715
Recall: 0.84375
F1: 0.8059701492537313
AUROC: 0.5546875


In [126]:
patient_oof.to_csv(
    "../data/processed/features/patient_rf_oof.csv",
    index=False
)

print("Saved patient RF OOF predictions.")
print("Shape:", patient_oof.shape)

Saved patient RF OOF predictions.
Shape: (44, 4)
